# D-algebraic functions expansion

In [1]:
import sys
sys.path.insert(0, "..") # dalgebra is here
from dalgebra import *
from dalgebra.pseries.laurent import *

%display latex

I want to think about what to do to detect possible orders for Laurent series expansions around 0 of solutions of D-algebraic equations. 

I am not fuly sure on all the steps but I have an intuition on how to proceed.

In order to work properly, I am going to need at least three examples:
* A linear example: things are easy here and a polynomial (indicial polynomial) can be computed.
* A purely D-algebraic case:
  - With guaranteed highest order monomial with its highest derivative
  - With highest order monomial without its highest derivative.
 
Then, it remains to check how to compute the expansion of the Laurent series solutions and how can we actually compute which inicial conditions are needed.

## Generating the power series solution

Given an equation, we may want to look when it has a "low order" solution. This requires to set up a given ansatz and then extend the possible solution. After that, we need to check the relations between the initial conditions to have an actual solution to the equation.

In [2]:
def generic_solution(equation, gen):
    R = equation.parent()
    C = R.constant_ring()
    order = equation.order(gen)

    nC = C.add_constants(*[f"a_{i}" for i in range(order)])
    a = nC.constant_ring().gens()
    return equation.power_series_solution(gen, {i: a[i] for i in range(order)})

## The linear case

In this case, we have an equation of the shape:
$$L = \alpha_0 u+ \alpha_1 u' + \ldots + \alpha_n u^{(n)}.$$
We assume the elements $\alpha_i \in C[[x]]$, so thei can only have a positive order.

Hence, each term will have order $d - i + ord(\alpha_i)$. This makes comparisons quite trivial and independent of $d$.

In [3]:
C = DifferentialRing(QQ)
R = DifferentialRing(QQ[x], [1])
R.set_constant(C)
F = R.fraction_field()
x, = F.gens()
DO.<u> = DifferentialPolynomialRing(F)
LS.<t> = LaurentSeries(C)
mor = F.laurent_morphism({'x': t}, set_default=True)

### 1.1. A monic case

$$L = u^{(5)} - 1/(1-x) u^{(3)} + 2xu'' - u$$

In [4]:
L = u[5] - (1/(1-x))*u[3] + 2*x*u[2] - u[0]
L
PR.<k> = PolynomialRing(ZZ)

In [5]:
def goal_order(mon,coeff,var):
    output = [0, None]
    for ((_,o),e) in mon._variables.items():
        output[0]+=e*var-o[0]
    output[0] += mor(coeff).order(bound=100)
    output[1] = mon
    return output

Here are the order for each term in the equation:

In [6]:
orders = [goal_order(m,c,k) for m,c in (zip(L.monomials(u), L.coefficients(u)))]

We would need that two of them are equal in order to have a cancellation. Hence:

In [7]:
candidates = []
for i in range(len(orders)):
    for j in range(i+1, len(orders)):
        ord_i = orders[i][0]
        ord_j = orders[j][0]
        candidates.append((ord_i - ord_j).roots())
candidates

[[], [], [], [], [], []]

Since we have zero candidates, then we conclude that the order of the series must be greater or equal than 0 and smaller than $5$.

We can check now a generic solution for relations among the initial conditions to be solutions of our equation:

In [16]:
sol = generic_solution(L, u)
mor_sol = L.parent().laurent_morphism(imgs={'u': sol}, constant=sol.parent().constant_ring())
L_eval = mor_sol(L)
L_eval.is_zero()

20

Since we get zero in the evaluation, we can conclude that up to order 20, we have no condition, hence, any possibility for the constants in the generic solution provide a solution to the equation.

This makes sense, because we are considering a linear differential equation, whose solutions are a C-vector space and the initial conditions guarantee a linear independence. 

### 1.2. The special case: Bessel differential equation

In [18]:
L = x^2*u[2] + x*u[1] + (x^2-25)*u[0]
orders = [goal_order(m,c,k) for m,c in (zip(L.monomials(u), L.coefficients(u)))]
candidates = []
for i in range(len(orders)):
    for j in range(i+1, len(orders)):
        to_root = orders[i][0] - orders[j][0]
        if to_root == 0:
            candidates.append(orders[i][0])
        else:
            candidates.append((ord_i - ord_j).roots())
candidates

[k, k, k]

In this case, we have an equation so homogeneous that any function with order different than $(0,1)$ may have cancellations. So these candidates are not good enough. Let us dive a bit deeper. If we assume a good order (i.e., different than $0,1$) then we have that 
$$[x^{k}] L\cdot u(x) = k(k-1)u_{k} + ku_{k} - 25u_k = (k^2 - k + k - 25) u_k = (k^2 - 25)u_k.$$
Since we assume the order of $u$ is $k$, then we know that $u_k \neq 0$. Hence we obtain $k \in \{\pm5\}$. These are the integer roots of the actual inidicial polynomial. This case is also quite interesting since extending the initial values is not as simple as it may seem using the classical approach of computing $u''$ from the equation and derivating here.

In this case, we use the following approach:
$$[x^p] L\cdot u(x) = [x^p] x^2u'' + [x^p] xu' + [x^p](x^2-25)u = [x^{p-2}] u'' + [x^{p-1}] u' + [x^{p-2}]u - 25 [x^p]u = p(p-1)u_p + pu_p + u_{p-2} - 25u_p = (p^2 - 25)u_p + u_{p-2},$$
so we can get the value:
$$u_{p} = \frac{u_{p-2}}{(25-p^2)},$$
which is a very simple formula tha works for any $p \notin\{\pm5\}$.

We could easily get this formulation due to the nature of the equation $L$: is is a D-finite equation (linear with polynomial coefficients) so it is known that the sequence of any solution is $P$-finite (or define with a linear recurrence with polynomial coefficients).

In [25]:
def _bessel_coeff(k,n):
    if k < n:
        return 0
    elif k == n:
        return 1
    else:
        return _bessel_coeff(k-2,n)/(n^2 - k^2)
sol = LS.element_class(LS, coefficient_map=lambda k : _bessel_coeff(k,5), order=5)

In [26]:
sol

t^5 - 1/24*t^7 + 1/1344*t^9 - 1/129024*t^11 + 1/18579456*t^13 - 1/3715891200*t^15 + 1/980995276800*t^17 - 1/329614413004800*t^19 + O(t^21)

In [47]:
f = Bessel(5)(SR('t'))

In [49]:
[f.derivative(i)(t=0) for i in range(20)]

[0,
 0,
 0,
 0,
 0,
 1/32,
 0,
 -7/128,
 0,
 9/128,
 0,
 -165/2048,
 0,
 715/8192,
 0,
 -3003/32768,
 0,
 1547/16384,
 0,
 -12597/131072]

2^5

In [54]:
(k^3 - 3*k^2 + 2*k+1).roots()

[]

In [55]:
(k^2 - 4*k+2).roots()

[]